In [ ]:
import pandas as pd

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination

# Step 1: Load the heart disease dataset
data = pd.read_csv("heart.csv")

# Handle different versions of the standard UCI dataset
data.columns = data.columns.str.strip().str.lower()

if "num" in data.columns:
    data.rename(columns={"num": "target"}, inplace=True)
elif "goal" in data.columns:
    data.rename(columns={"goal": "target"}, inplace=True)

# Replace missing-value markers and convert columns to numeric
data.replace("?", pd.NA, inplace=True)

columns = [
    "age", "sex", "cp", "trestbps", "chol", "fbs",
    "restecg", "thalach", "exang", "oldpeak",
    "slope", "ca", "thal", "target"
]

data = data[columns]

for col in columns:
    data[col] = pd.to_numeric(data[col], errors="coerce")

# Remove rows containing missing values
data.dropna(inplace=True)

# Convert target values into two classes:
# 0 = No heart disease, 1 = Heart disease
data["target"] = (data["target"] > 0).astype(int)

# Step 2: Convert continuous values into categories
# Bayesian networks in pgmpy use discrete states here.

data["age"] = pd.cut(
    data["age"], bins=[0, 40, 50, 60, 100],
    labels=["young", "middle", "senior", "elderly"]
)

data["trestbps"] = pd.cut(
    data["trestbps"], bins=[0, 120, 140, 160, 300],
    labels=["normal", "elevated", "high", "very_high"]
)

data["chol"] = pd.cut(
    data["chol"], bins=[0, 200, 240, 300, 1000],
    labels=["normal", "borderline", "high", "very_high"]
)

data["thalach"] = pd.cut(
    data["thalach"], bins=[0, 100, 140, 170, 250],
    labels=["low", "medium", "high", "very_high"]
)

data["oldpeak"] = pd.cut(
    data["oldpeak"], bins=[-1, 0, 1, 2, 10],
    labels=["none", "mild", "moderate", "severe"]
)

# Remove rows that could not be placed into a category
data.dropna(inplace=True)

# Convert categorical values into strings
for col in data.columns:
    data[col] = data[col].astype(str)

data["target"] = data["target"].astype(int)

# Step 3: Construct the Bayesian Network structure
# Each edge represents a dependency between two variables.

model = DiscreteBayesianNetwork([
    ("age", "trestbps"),
    ("age", "thalach"),
    ("sex", "thalach"),
    ("cp", "exang"),
    ("cp", "target"),
    ("trestbps", "target"),
    ("chol", "target"),
    ("fbs", "target"),
    ("restecg", "target"),
    ("thalach", "target"),
    ("exang", "target"),
    ("oldpeak", "target"),
    ("slope", "target"),
    ("ca", "target"),
    ("thal", "target")
])

# Step 4: Learn conditional probabilities from the dataset
model.fit(
    data,
    estimator=BayesianEstimator,
    prior_type="BDeu",
    equivalent_sample_size=10
)

# Step 5: Perform Bayesian inference
inference = VariableElimination(model)

# Example patient details (already converted into categories)
patient = {
    "age": "senior",
    "sex": "1",
    "cp": "2",
    "trestbps": "high",
    "chol": "high",
    "fbs": "0",
    "restecg": "0",
    "thalach": "medium",
    "exang": "1",
    "oldpeak": "moderate",
    "slope": "2",
    "ca": "1",
    "thal": "3"
}

# Calculate probability of each diagnosis
result = inference.query(
    variables=["target"],
    evidence=patient
)

# Display the diagnosis probabilities
print("\nHeart Disease Diagnosis:")
print(result)

# Display the most probable outcome
probability = result.values
disease_probability = probability[1] * 100

print("\nProbability of Heart Disease:", round(disease_probability, 2), "%")

if disease_probability >= 50:
    print("Predicted Result: Heart Disease")
else:
    print("Predicted Result: No Heart Disease")